# Rapport TP Middleware

## TP-10 Skill Réservation

### 3. Gestionnaire de ressources

**Quel est le nom du gestionnaire de ressources de la plateforme Grid'5000 ?**

Le nom du gestionnaire de ressources est OAR.

**Quelle est la commande permettant d'effectuer une réservation de ressources ?**

```bash
oarsub -I
```

**Quelle est la différence entre les options « -I » et « -t » ?**

`-I` est le mode interactif, permettant de communiquer avec le nœud. `-t` permet une configuration plus spécifique du nœud grâce à plusieurs options disponibles.

### 4. Réserver un nœud en mode interactif

**Réservez un nœud de calcul en mode interactif pendant une heure :**

```bash
oarsub -I walltime = 1:00:00
```

**Que se passe-t-il après l'exécution de la commande ?**

OAR filtre les ressources exotiques, définit le walltime et alloue un nœud avec un ID unique, permettant de se connecter à une machine en mode interactif.

**Comment afficher les détails de la réservation ?**

```bash
lscpu
```

### 5. Réserver un nœud spécifique

**Écrire la commande permettant de réserver 2 nœuds sur le site de Nantes pendant 20 minutes :**

```bash
ssh nantes
oarsub -l nodes=2,walltime=0:20 -I
```

**Que signifie l'option -p ?**

L'option `-p` permet de choisir un cluster précis, par exemple `-p taurus`.

**Que se passe-t-il si le cluster est mal orthographié ?**

```
Bad resource request (ERROR: column "econom" does not exist)
```

**Que fait la commande `oarstat -u` ?**

Elle donne la liste des jobs en cours d'utilisation.

### 6. Réservation avec un script de job

**Soumettre le job en réservant un nœud pendant 10 minutes sur le site de Rennes :**

```bash
ssh rennes
oarsub -l walltime=0:10 -I          # sortie affichée dans le terminal
# ou
oarsub -l walltime=0:10 "./mon_script.sh"
```

**Où est redirigée la sortie du script ?**

Dans le terminal si le job est lancé en mode `-I`, ou dans les fichiers `OAR..stdout` / `OAR..stderr` sinon.

**Que contiennent les fichiers `OAR..stdout` et `OAR..stderr` ?**

Le `stdout` contient la sortie du programme `.sh`, et le `stderr` donne :
```
/var/lib/oar/.batch_job_bashrc: line 5: /home/cchiaber/.bashrc: No such file or directory
```

**Que se passe-t-il si le script plante ?**

Il n'y a pas de fichiers `OAR..stdout` et `OAR..stderr` créés.

---

## TP-11 Skill NodeJS

### 2. Exécution de la page Hello World

```
cchiaber@econome-22:~/public/TP-10_11$ node hello-world.js
Hello from the class => 'Middleware 2024'
Hello again from the class => Middleware 2024
constItem => 10

letItem => I love coffee with 2 slices of blueberry cake !
letItem => I love coffee with 3 slices of blueberry cake !
letItem => I love coffee with 4 slices of blueberry cake !

varItem => I love to drive but only up to 1.5 hours and 1 second !
varItem => I love to drive but only up to 2 minutes !
varItem => I love to drive but only up to 3 minutes !
varItem => I love to drive but only up to 4 minutes !
varItem => I love to drive but only up to 5.5 hours and 1 second !
```

### 3. Installation des packages Node.js et exécution du script principal

```
cchiaber@econome-22:~/public/TP-10_11$ node main-express-static.js
Listening to 172.16.192.22:3000 !!!
```

### 4. Redirection du port 3000 vers la machine locale

```
cchiaber@fnantes:/$ ssh -NL 3000:172.16.192.22:3000 cchiaber@access.grid5000.fr
Warning: Permanently added 'access.grid5000.fr' (ED25519) to the list of known hosts.
bind [127.0.0.1]:3000: Address already in use
channel_setup_fwd_listener_tcpip: cannot listen to port: 3000
Could not request local forwarding.
```

![Redirection du port 3000](redirection_port3000.png)

---

## TP-12 Générateur de données

`main-data-generator` est un petit serveur qui écoute le port 3000, configure la génération, génère les données aléatoires et les stocke, attend et répond aux requêtes GET/POST via HTML/JSON.

`test-data-generator` se comporte comme un client et envoie des requêtes POST au serveur.

**Quels sont les rôles des paquets `express`, `ip`, `clone`, `ejs` et `axios` ?**

- **express** : ici, un framework Node.js permettant de simplifier la communication HTTP (les routes, les vues, etc.), côté serveur.
- **ip** : permet de récupérer l'adresse IP locale.
- **clone** : permet de faire une copie profonde d'un objet JavaScript.
- **ejs** : permet de générer du HTML.
- **axios** (dans `test-data-generator`) : permet d'envoyer des requêtes HTTP, côté client.

**Quelles sont les différentes commandes effectuées par le fichier de test ?**

Les commandes effectuées sont des requêtes AJAX :

- **Setting** : permet de choisir le type de génération
- **DeleteAllData** : vide l'historique des données
- **Start** : démarre la génération
- **Stop** : stop la génération
- **Fetch-Data** : récupère toutes les données générées entre le Start et le Stop

![Page web du générateur de données](webpageHTML.png)

**Ouvrez le fichier `webpage-data-generator.ejs`, pour en comparer le contenu avec `webpage-data-generator.html`.**

Le contenu qu'on retrouve dans le `.html` est bien celui attendu après exécution du `.ejs`.

On y retrouve le même template, les mêmes informations (titre : `<h1>Bienvenue sur le générateur de données!</h1>`, tableau : `<h2>Données</h2> <table...>`).

Et les données générées et récupérées par le serveur, insérées dans le tableau par :

```ejs
<% htmlDataPoints.forEach( dp=> { %>
    <tr>
        <th style="border-style:solid;border-width:1px;">
            <%= dp['TimeStampFormatted'] %>
        </th>
        <th style="border-style:solid;border-width:1px;">
            <%= dp['DataValue'] %>
        </th>
    <% }) %>
</tr>
```

```
cchiaber@econome-8:~/public/TP-12-Générateur de données$ node main-data-generator.js 
Listening to 172.16.192.8:3000 !!!
```

> `ssh econome-xx.nantes.grid5000.fr` pour se connecter à un nœud en particulier d'un cluster.

```
cchiaber@econome-8:~/public/TP-12-Générateur de données$ node test-data-generator.js 
{ Message: 'Config/Update: ' }
{ Message: 'Execute:DeleteAllData' }
{ Message: 'Execute:Start' }
{ Message: 'Execute:Stop' }
[
  {
    TimeStampRaw: '2026-09-11T14:47:40.637Z',
    TimeStampFormatted: '2026-09-11 16:47:40.637',
    DataValue: 0
  }
]
```

![Premier lancement](images/1erLancement.png)

Étant donné que les algorithmes de génération ne sont pas écrits, ils retournent 0.

### Tâche

Compléter le fichier `main-data-generator.js` pour :

- générer des nombres aléatoires selon deux distributions (uniforme et normale) (lignes 58 à 66)
- permettre un affichage à l'aide des commandes Start, Stop et DeleteAll (lignes 75 à 97)

### Rapport

**Mise en place des méthodes de distributions des données**

Pour la distribution uniforme, chaque valeur a exactement la même probabilité d'être générée.
Soit `Params: { min: -10, max: 10 }`.

Pour la distribution normale, elle est centrée autour d'une moyenne (mu), avec un écart-type (sigma) qui contrôle la dispersion des valeurs autour de mu.
Soit `Params: { mu: 100, sigma: 10 }`.

L'intervalle de temps entre les deux générations restera le même pour les deux types de génération.

Le package `random` implémente déjà ces distributions : `random.uniform(min, max)` et `random.normal(mu, sigma)`.

Les paramètres `min`, `max` et `mu`, `sigma` peuvent être récupérés via `params`, qui dépend du `DistributionType` appliqué après la commande Setting, donc récupérable dans le JSON généré.

**Implémentation des commandes Start, Stop et DeleteAll**

Pour finir l'implémentation de Start, j'ai rajouté le lancement de la fonction `DataGeneration` avec l'intervalle « Interval » donné dans la configuration du serveur.

Pour implémenter la commande Stop, j'ai relancé `clearInterval(timerDataGenerator)` qui stoppe la génération en cours.

Pour la commande DeleteAll, j'ai réalloué le tableau avec `arrRandomNumbers.length = 0` pour le remettre à zéro.

```
chiaber@econome-22:~/public/TP-12-Générateur de données$ node main-data-generator.js 
Listening to 172.16.192.22:3000 !!!
TypeError: random.uniform is not a function
    at /home/cchiaber/public/TP-12-Générateur de données/main-data-generator.js:68:52
    at Layer.handleRequest (/home/cchiaber/node_modules/router/lib/layer.js:152:17)
    at next (/home/cchiaber/node_modules/router/lib/route.js:157:13)
    at Route.dispatch (/home/cchiaber/node_modules/router/lib/route.js:117:3)
    at handle (/home/cchiaber/node_modules/router/index.js:435:11)
    at Layer.handleRequest (/home/cchiaber/node_modules/router/lib/layer.js:152:17)
    at /home/cchiaber/node_modules/router/index.js:295:15
    at processParams (/home/cchiaber/node_modules/router/index.js:582:12)
    at next (/home/cchiaber/node_modules/router/index.js:291:5)
    at serveStatic (/home/cchiaber/node_modules/serve-static/index.js:74:16)
/home/cchiaber/public/TP-12-Générateur de données/main-data-generator.js:86
                        x1 = funcDataGenerator()
                             ^

TypeError: funcDataGenerator is not a function
    at Timeout._onTimeout (/home/cchiaber/public/TP-12-Générateur de données/main-data-generator.js:86:30)
    at listOnTimeout (node:internal/timers:581:17)
    at process.processTimers (node:internal/timers:519:7)

Node.js v20.20.2
```

```js
case 'Normal':
    //Ecrire cette section
    funcDataGenerator = random.normal(
        jsonSetting['DataGeneration']['Params']['mu'],
        jsonSetting['DataGeneration']['Params']['sigma']
    )
    break

case 'Uniform':
     //Ecrire cette section
    funcDataGenerator = random.uniform(
        jsonSetting['DataGeneration']['Params']['min'],
        jsonSetting['DataGeneration']['Params']['max']
    )
    break
```

Cette implémentation renvoie des valeurs et non une fonction dans `funcDataGenerator`.

Pour régler ce problème, j'ai fait des fonctions anonymes pour forcer le type : `funcDataGenerator = () => random.normal(...)`.

![Échec de la génération](failed2.png)

Toutes les valeurs sont nulles, `x1` récupère autre chose qu'un nombre. Après avoir demandé à un camarade, il suffisait d'appliquer `const random = require('random').default` pour résoudre le problème `TypeError: funcDataGenerator is not a function` ; `funcDataGenerator` reprend alors les valeurs renvoyées par `random.normal()` et `random.uniform()`.

On reçoit alors :

**Uniforme**

![Distribution uniforme réussie](win.png)

**Normale**

![Distribution normale réussie](winNormal.png)